### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [22]:
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# ! vectorstores
from langchain_community.vectorstores import FAISS, Chroma, Pinecone, Weaviate, Milvus, Qdrant


## utility imports
import numpy as np
from typing import List

In [23]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [24]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    eff

In [25]:
import tempfile
temp_dir = tempfile.mkdtemp()
print(f"Temporary directory created at: {temp_dir}")
for i, doc in enumerate(sample_docs):
    file_path = f"docs/doc_{i+1}.txt"
    with open(file_path, 'w') as f:
        f.write(doc)
    print(f"Document {i+1} saved at: {file_path}")
    

Temporary directory created at: C:\Users\ARNAVB~1\AppData\Local\Temp\tmpolatbwtq
Document 1 saved at: docs/doc_1.txt
Document 2 saved at: docs/doc_2.txt
Document 3 saved at: docs/doc_3.txt


In [26]:
from langchain_community.document_loaders import TextLoader,DirectoryLoader

loader=DirectoryLoader("docs", glob="*.txt", )
documents=loader.load()
print(f"Number of documents loaded: {len(documents)}")
print("Sample document content:")
print(documents[0].page_content[:500])  # Print the first 500 characters of the first document


Number of documents loaded: 3
Sample document content:
Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewar


## Document Splitting

In [27]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50,length_function=len)
# text_splitter.split_documents(documents)
chunks=text_splitter.split_documents(documents)
print(f"Number of chunks created: {len(chunks)}")
print("Sample chunk content:")
print(chunks[0].page_content[:500])  # Print the first 500 characters of the first chunk

Number of chunks created: 5
Sample chunk content:
Machine Learning Fundamentals


In [28]:
chunks

[Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'docs\\doc_2.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'docs\\doc_2.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep le

## Embedding MOdels

In [31]:
sample_text="Machine Learning is facinating"
embeddings=OllamaEmbeddings(model="llama3.2:latest")
embeddings


OllamaEmbeddings(model='llama3.2:latest', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [32]:
embeddings.embed_query(sample_text)

[-0.012043369,
 0.016861685,
 0.04180385,
 -0.014466409,
 -0.020095624,
 -0.04560012,
 0.018800316,
 0.0004905143,
 -0.013566924,
 -0.015364271,
 -0.008641946,
 -0.020964768,
 0.0074825077,
 0.03223642,
 0.0058638556,
 -0.011616594,
 0.008150148,
 0.0023836666,
 0.00086505245,
 0.0061067375,
 0.0021974372,
 -0.021860462,
 0.029815856,
 -0.017236723,
 0.0076064547,
 -0.0153949605,
 0.023596995,
 -0.040058948,
 0.017899202,
 0.004671267,
 -0.009727652,
 -0.006471602,
 0.009034995,
 -0.0053049237,
 0.03227416,
 0.00048502005,
 -0.015607403,
 0.028760463,
 -0.0101156235,
 -0.023348317,
 -0.018770183,
 -0.01744114,
 0.011482883,
 0.0046852655,
 -0.018003376,
 0.0052916748,
 0.0066463216,
 0.014486938,
 0.011929245,
 -0.02734766,
 0.0077267885,
 0.017156003,
 0.020057231,
 0.010966691,
 0.0014332853,
 0.006344407,
 0.015087748,
 -0.028868567,
 0.007870271,
 0.019556755,
 0.0065760785,
 0.0037592635,
 0.026607724,
 -0.0064209765,
 0.0298617,
 -0.0754462,
 -0.017583137,
 -0.0062064254,
 -0.004

## Initialize the Chroma DB

In [33]:
persistance_directory="./chroma_db"
chroma_db=Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory=persistance_directory,collection_name="rag_collection")

print("Persisting ChromaDB to disk...")
chroma_db.persist()
print(f"ChromaDB persisted successfully at: {persistance_directory}")
print(chroma_db._collection.count())


Persisting ChromaDB to disk...
ChromaDB persisted successfully at: ./chroma_db
5


C:\Users\ArnavBhatia\AppData\Local\Temp\ipykernel_59988\4266146322.py:5: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_db.persist()


Similarity Search

In [34]:
query="What is machine learning?"
similar_docs=chroma_db.similarity_search(query,k=2)
print(f"Number of similar documents retrieved: {len(similar_docs)}")
similar_docs


Number of similar documents retrieved: 2


[Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine Learning Fundamentals')]

In [35]:
query="What is NLP?"
similar_docs=chroma_db.similarity_search(query,k=2)
print(f"Number of similar documents retrieved: {len(similar_docs)}")


Number of similar documents retrieved: 2


In [36]:
query="What is Deep Learining?"
similar_docs=chroma_db.similarity_search(query,k=2)
print(f"Number of similar documents retrieved: {len(similar_docs)}")


Number of similar documents retrieved: 2


In [37]:
for i ,doc in enumerate(similar_docs):
    print(f"Similar Document {i+1} Content:")
    print(doc.page_content)
    print("-" * 50)
    print(f"Metadata: {doc.metadata}")

Similar Document 1 Content:
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.
--------------------------------------------------
Metadata: {'source': 'docs\\doc_1.txt'}
Similar Document 2 Content:
Deep Learning and Neural Networks
--------------------------------------------------
Metadata: {'source': 'docs\\doc_2.txt'}


### Advance Similarity Search

In [38]:
results_scores=chroma_db.similarity_search_with_score(query,k=2)
results_scores

[(Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'),
  0.7129333372988603),
 (Document(metadata={'source': 'docs\\doc_2.txt'}, page_content='Deep Learning and Neural Networks'),
  0.7519029369195995)]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

### Initialize LLM,RAG,Prompt Technique, Query the RAG System

In [39]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:latest")
llm.invoke("Hello, how are you?")


AIMessage(content="I'm just a language model, so I don't have feelings in the way that humans do, but I'm functioning properly and ready to help with any questions or tasks you may have. How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-24T16:40:50.2126482Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1143018300, 'load_duration': 210266100, 'prompt_eval_count': 31, 'prompt_eval_duration': 241072500, 'eval_count': 46, 'eval_duration': 585776500, 'model_name': 'llama3.2:latest'}, id='lc_run--019dc05d-8a3d-7713-b628-731c6926f9d4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 46, 'total_tokens': 77})

In [40]:
test_response=llm.invoke("What is LLM?")
test_response

AIMessage(content="LLM stands for Large Language Model. It's a type of artificial intelligence (AI) model designed to process and understand human language at a massive scale.\n\nLarge Language Models are typically trained on vast amounts of text data, which allows them to learn patterns, relationships, and structures within language. This training enables the model to generate coherent and contextually relevant responses, often indistinguishable from those produced by humans.\n\nLLMs are commonly used in natural language processing (NLP) applications, such as:\n\n1. Language translation\n2. Text summarization\n3. Sentiment analysis\n4. Chatbots and virtual assistants\n5. Content generation\n\nThe key characteristics of Large Language Models include:\n\n1. **Massive size**: LLMs are typically composed of billions of parameters (e.g., 175 billion in the case of BERT) and process vast amounts of data.\n2. **Deep learning architecture**: LLMs use deep neural networks, which consist of mul

In [43]:
from langchain.chat_models.base import init_chat_model
llm = init_chat_model("ollama:llama3.2:latest")
llm.invoke("What is RAG?")

AIMessage(content="RAG can refer to several things, but here are a few possible interpretations:\n\n1. Recycled Automotive Grade (RAG): A term used to describe recycled materials that meet the standards of automotive manufacturers for use in vehicle production.\n2. Rag and Bone: A brand name that originated from an 18th-century textile company in London. It's now known for its streetwear-inspired clothing, accessories, and home goods.\n3. Raggedy Ann: A popular children's doll character created by Johnny Gruelle in the early 20th century.\n4. RAGBRAI (Register's Annual Great Bicycle Ride Across Iowa): An annual bicycle ride that takes place in the United States, typically held in July.\n\nIf you could provide more context or information about which RAG you're referring to, I'd be happy to try and give a more specific answer!", additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-04-24T16:42:45.2890311Z', 'done': True, 'done_reason': 'stop', 'total_du

##  Modern Rag Chain 

In [55]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

retriever=chroma_db.as_retriever(search_kwargs={"k":3})
system_prompt="You are a helpful assistant that provides concise answers based on retrieved documents. {context}"
prompt=ChatPromptTemplate.from_messages([("system",system_prompt),("human","{input}")])

combine_docs_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)
print(combine_docs_chain)



bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a helpful assistant that provides concise answers based on retrieved documents. {context}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOllama(model='llama3.2:latest')
| StrOutputParser() kwargs={} config={'run_name': 'stuff_documents_chain'} config_factories=[]


This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

retrieval

In [57]:
retrieval_chain=create_retrieval_chain(retriever,combine_docs_chain)
query="What is NLP?"
response=retrieval_chain.invoke({"input": query})

In [60]:
print("Final Response:")
for i,j in response.items():
    print(f"{i}: {j}")


Final Response:
input: What is NLP?
context: [Document(metadata={'source': 'docs\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, while unsupervised learning finds patterns in unlabeled data. Reinforcement learning learns through interaction with an environment using rewards and penalties.'), Document(metadata={'source': 'docs\\doc_3.txt'}, page_content='Natural Language Processing (NLP)\n\nNLP is a field of AI that focuses on the interaction between computers and human language. Key tasks in NLP include text classification, named entity recognition, sentiment analysis, machine translation, and question answering. Modern NLP heavily relies on transformer architectures like BERT, GPT, an

In [49]:
import langchain
langchain.__version__

'1.2.12'